# 🎯 [QLT04_M02] 정비 대기 시간(부품 대기, 정비병 부족, 외주 정비) 요인 분류 전처리 모듈 구현 (훈련생 실습용)
### 훈련생 실습 워크북 (Google Colab / Jupyter Student Edition)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/QLT04/qlt04_m02_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 품질·규격·검사 (Level 2) |
| **미션 ID** | QLT04_M02 |
| **미션 주제** | 정비 대기 시간(부품 대기, 정비병 부족, 외주 정비) 요인 분류 전처리 모듈 구현 |
| **소요 시간** | 약 2시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
외주정비는 정비 대기, 외주 검사 대기, 재작업, 규격·제원 불일치가 겹치면서 장비 가동률을 떨어뜨리지만 요인별 영향이 한눈에 보이지 않아 어디에 정비 자원을 먼저 투입할지 정하기 어렵다. 본 프로젝트는 미 국방부 외주정비 품질보증 사례의 문제의식(규격 정합성·검사 병목·하자 연결)을 교육용 가상 자료로 재구성해, 정비 대기 요인 분류, 저가동 요인 분석, 재배치 시뮬레이션, 추세 대시보드, 사람 검토 워크플로우까지 프로토타입으로 구현하는 과제이다. 가상 장비는 유형 A·B·C와 가상 부대로만 구분하며 실제 장비·부대 자료는 쓰지 않는다. 최종 목표는 가동률 저하 요인과 외주 품질검사 병목을 근거와 함께 보여 주되, 정비 우선순위와 자원 투입 결정은 정비관·품질검사관·군수참모가 하도록 지원하는 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

> **💡 [오프라인 폐쇄망 및 로컬 주피터 환경 안내]**
> - 외부 인터넷망(Colab) 외에 군 인트라넷 또는 로컬 환경에서 실행할 경우, 아래 명령어로 필요한 표준 라이브러리를 사전 설치하십시오:
>   `pip install numpy pandas matplotlib scipy scikit-learn`
> - 인터넷이 제한된 환경의 경우 사전 번들링된 Wheel(`.whl`) 패키지 디렉토리를 통해 설치할 수 있습니다:
>   `pip install --no-index --find-links=./wheels/ -r requirements.txt`

> 📁 **[안내: 프로젝트 표준 산출물 디렉터리 가이드]**
> - 본 프로젝트는 단순한 일회성 코드 실습을 넘어 실제 부대에 배포 가능한 업무 도구(Tool)를 완성하는 과정입니다.
> - 실습 중 작성하는 결과물은 역할에 따라 아래 표준 디렉터리에 자동으로 저장·누적됩니다:
>   - `docs/`: 프로세스 정의서(`qlt04_as_is_to_be_spec.md`), UAT 피드백 리포트(`qlt04_uat_feedback_report.md`), 운영 매뉴얼(`qlt04_final_manual.md`) 등 행정·기술 문서
>   - `src/`: 데이터 로더, 16종 품질 판정 룰 엔진, 릿지 회귀 XAI 엔진, 시뮬레이터 UI, HITL 결재 FSM, 배치 러너 등 핵심 실행 모듈
>   - `dist/`: 폐쇄망 및 야전 배포를 위한 최종 패키지(`qlt04_package/`)

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 02. 정비 대기 시간(부품 대기, 정비병 부족, 외주 정비) 요인 분류 전처리 모듈 구현
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

정비 대기 시간(부품 대기, 인력 부족) 요인 분류 전처리 모듈을 구현한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **서식이 다른 정비 대기 기록(CSV, 엑셀 내보내기, JSON, 정비일지 문장)을 하나의 표준 스키마로 바꾸고 대기 사유를 부품 대기·정비병 부족·외주 정비·검사·재작업으로 분류하는 전처리 모듈을 구현한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **열 이름 별칭 사전으로 CSV, 탭 구분(엑셀 내보내기), JSON 세 형식을 표준 스키마로 바꾸고, 대기 기간(일·주·개월·시간)과 날짜·부대명 표기를 통일 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/connectors/qlt04_data_loader.py 및 스키마 정의서` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 국방 서식/양식 변경에 대한 유연한 확장성

#### 미션 PASS 조건

- 정량 기준: 지원 파일 포맷(CSV, Excel, JSON 등) 파싱 성공률 100%


- 진행 기준: 데이터 로더 및 스키마 유효성 검사 모듈 완성


- 검증 방법: 다양한 형태의 샘플 군수 문서 대상 파싱 테스트


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""서식이 서로 다른 가상 정비 대기 기록(CSV, 엑셀 내보내기 TSV, JSON, 정비일지 문장)을 열 이름 별칭 사전과 정규식으로 표준 스키마로 바꾸고, 대기 일수·부대명·날짜 표기를 통일하며 대기 사유를 부품 대기·정비병 부족·외주 정비·검사·재작업으로 분류하는 전처리 모듈을 구현한다."""
import csv
import io
import json
import re
from datetime import date
import pandas as pd

# ---------------------------------------------------------------------------
# [교육용가상자료] 서로 다른 형식의 원시 기록 (실제 부대·장비 아님). 표기가 일부러 제각각이다.
# ---------------------------------------------------------------------------
RAW_CSV = """장비,부대,대기사유,대기기간,접수일
E-0001,가상A부대,부품 미입고,5일,2026.03.02
E-0002,가상 B 부대,정비병 부족(휴가·교육),2주,2026-3-15
E-0003,C부대,외주업체 입고 대기,48시간,20260110
E-0004,가상D부대,외주 검사 대기,3일,26/02/27
E-0005,가상E부대,부품 청구 지연,1개월,2025.12.30
E-0006,가상F부대,재작업(검사 불합격),4일,2026-01-05
"""
RAW_TSV = "장비번호\t소속\t사유\t지연일수\t접수\n" \
          "E-0007\t가상A부대\t정비인력 부족\t6\t2026.02.11\n" \
          "E-0008\t가상B부대\t부품 입고 지연\t10\t2026.01.20\n" \
          "E-0009\t가상C부대\t외주 정비 수행 중\t1주\t2026/03/05\n" \
          "E-0010\tD부대\t규격 불일치로 재검사\t3\t2025.11.03\n" \
          "E-0001\t가상A부대\t부품 미입고\t5일\t2026.03.02\n"
RAW_JSON = json.dumps({"result": "OK", "items": [
    {"equip": "E-0011", "unit": {"name": "가상E부대"}, "reason": "예비품 청구 후 입고 대기", "wait": {"value": 2, "unit": "week"}, "received": "2026-02-01"},
    {"equip": "E-0012", "unit": {"name": "가상F부대"}, "reason": "정비병 교육 파견", "wait": {"value": 4, "unit": "day"}, "received": "2026-01-09"},
    {"equip": "E-0013", "unit": {"name": "가상A부대"}, "reason": "외주 검사 일정 대기", "wait": {"value": 72, "unit": "hour"}, "received": "2026-03-20"}]},
    ensure_ascii=False)
LOGS = [
    "[정비일지] 2026-03-04 E-0021(가상A부대) 부품 입고 지연으로 6일 대기.",
    "가상B부대 E-0022 : 정비병 인력 부족, 3일간 착수 못함 (2026.03.05)",
    "E-0023 / 외주 정비 입고 대기 2주 / 가상C부대 / 26.03.06",
    "3월 7일자 가상D부대 E-0024 규격 불일치로 재검사 5일",
    "가상E부대 E-0025 원인 불명 대기 4일(2026-03-08)",
]
# 정답(분류 정확도 확인용): 기록 순서대로
TRUTH = {"E-0001": "부품 대기", "E-0002": "정비병 부족", "E-0003": "외주 정비", "E-0004": "검사·재작업", "E-0005": "부품 대기", "E-0006": "검사·재작업",
         "E-0007": "정비병 부족", "E-0008": "부품 대기", "E-0009": "외주 정비", "E-0010": "검사·재작업", "E-0011": "부품 대기", "E-0012": "정비병 부족",
         "E-0013": "검사·재작업", "E-0021": "부품 대기", "E-0022": "정비병 부족", "E-0023": "외주 정비", "E-0024": "검사·재작업", "E-0025": "기타"}

# ---- 정규화 규칙(서식이 바뀌면 이 사전만 고친다) ----
COL_ALIAS = {"equip": ["장비", "장비번호", "equip"], "unit": ["부대", "소속", "unit"], "reason": ["대기사유", "사유", "reason"],
             "wait": ["대기기간", "지연일수", "wait"], "received": ["접수일", "접수", "received"]}
UNIT_DAYS = {"일": 1, "일간": 1, "day": 1, "주": 7, "week": 7, "개월": 30, "month": 30, "시간": 1 / 24, "hour": 1 / 24}
# 분류 규칙: 위에서부터 먼저 걸리는 사유로 분류(검사·재작업 -> 외주 정비 -> 정비병 -> 부품)
CAUSE_KW = [("검사·재작업", ["검사", "재작업", "불합격", "규격 불일치"]), ("외주 정비", ["외주", "업체 입고"]),
            ("정비병 부족", ["정비병", "정비인력", "인력 부족", "휴가", "교육"]), ("부품 대기", ["부품", "미입고", "청구", "입고 지연"])]
DEFAULT_YEAR = 2026


def norm_unit(s):
    m = re.search(r"([A-F])\s*부대", str(s))
    if not m:
        raise ValueError(f"부대명 해석 불가: {s}")
    return f"가상{m.group(1)}부대"


def norm_wait(v):
    """대기 기간을 일 단위 숫자로 통일(일·주·개월·시간)."""
    if isinstance(v, dict):
        return round(v["value"] * UNIT_DAYS[v["unit"]], 1)
    m = re.match(r"^\s*(\d+)\s*(일간|일|주|개월|시간)?\s*$", str(v))
    if not m:
        raise ValueError(f"대기 기간 해석 불가: {v}")
    return round(int(m.group(1)) * UNIT_DAYS.get(m.group(2) or "일", 1), 1)


def norm_date(s):
    s = str(s).strip().rstrip(".")
    m = re.match(r"^(\d{2,4})[.\-/년\s]*(\d{1,2})[.\-/월\s]*(\d{1,2})일?$", s) or re.match(r"^(\d{4})(\d{2})(\d{2})$", s)
    if not m:
        m2 = re.match(r"^(\d{1,2})월\s*(\d{1,2})일", s)
        if m2:
            return date(DEFAULT_YEAR, int(m2.group(1)), int(m2.group(2))).isoformat()
        raise ValueError(f"날짜 해석 불가: {s}")
    y, mo, d = (int(x) for x in m.groups())
    return date(y + 2000 if y < 100 else y, mo, d).isoformat()


def classify(text):
    """
    [TODO : 학생 실습 2] 정비일지 문장(비정형 텍스트)을 분석하여 4대 지연 요인 중 하나로 분류하세요.
    - 힌트 1: CAUSE_KW의 (요인명, 키워드리스트)를 순회하며 text에 키워드가 포함되어 있는지 검사합니다.
    - 힌트 2: 매칭되는 키워드가 있으면 해당 요인명('부품대기', '정비병부족', '외주정비', '외주검사')을 반환합니다.
    - 힌트 3: 어떤 키워드에도 매칭되지 않는 경우 수기 검토 라우팅을 위해 '기타'를 반환합니다.
    """
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.


def pick(row, key):
    for k in COL_ALIAS[key]:
        if k in row:
            return row[k]
    raise KeyError(f"필수 열 없음: {key}")


def to_std(row, src):
    reason = str(pick(row, "reason"))
    return {"equip_id": str(pick(row, "equip")).strip(), "unit": norm_unit(pick(row, "unit")), "wait_days": norm_wait(pick(row, "wait")),
            "date": norm_date(pick(row, "received")), "reason": reason, "cause": classify(reason), "source": src}


def load(text, fmt):
    """형식별 로더: csv / tsv(엑셀 내보내기) / json. 행마다 성공·실패를 따로 센다."""
    if fmt == "json":
        rows = [{"equip": r["equip"], "unit": r["unit"]["name"], "reason": r["reason"], "wait": r["wait"], "received": r["received"]} for r in json.loads(text)["items"]]
    else:
        rows = list(csv.DictReader(io.StringIO(text), delimiter="\t" if fmt == "tsv" else ","))
    ok, fail = [], []
    for r in rows:
        try:
            ok.append(to_std(r, fmt))
        except (KeyError, ValueError) as e:
            fail.append((r, str(e)))
    return ok, fail


def parse_log(line):
    """정비일지 문장 1줄에서 장비·부대·대기 일수·날짜·사유를 추출한다."""
    eq = re.search(r"E-\d{4}", line)
    unit = re.search(r"(?:가상\s*)?[A-F]\s*부대", line)
    wait = re.search(r"(\d+)\s*(일간|일|주|개월|시간)", line)
    dt = re.search(r"(\d{2,4}[.\-/]\d{1,2}[.\-/]\d{1,2})|(\d{1,2}월\s*\d{1,2}일)", line)
    if not (eq and unit and wait and dt):
        raise ValueError("필수 항목(장비·부대·대기·날짜) 추출 실패")
    reason = re.sub(r"E-\d{4}|\[정비일지\]|\(.*?\)|\d{2,4}[.\-/]\d{1,2}[.\-/]\d{1,2}|\d{1,2}월\s*\d{1,2}일자?|가상\s*[A-F]\s*부대|[/:]", " ", line)
    reason = re.sub(r"\s+", " ", reason).strip()
    return {"equip_id": eq.group(0), "unit": norm_unit(unit.group(0)), "wait_days": norm_wait(wait.group(0)), "date": norm_date(dt.group(0)),
            "reason": reason, "cause": classify(reason), "source": "log"}


# 1) 형식별 파싱
results, total, success = [], 0, 0
for fmt, text in [("csv", RAW_CSV), ("tsv", RAW_TSV), ("json", RAW_JSON)]:
    ok, fail = load(text, fmt)
    results += ok; total += len(ok) + len(fail); success += len(ok)
    print(f"[{fmt.upper():4s}] {len(ok) + len(fail)}행 중 {len(ok)}행 성공" + (f", 실패 {[e for _, e in fail]}" if fail else ""))
logs, l_fail = [], 0
for t in LOGS:
    try:
        logs.append(parse_log(t))
    except ValueError as e:
        l_fail += 1; print("일지 파싱 실패:", e)
results += logs; total += len(LOGS); success += len(logs)
print(f"[정비일지 문장] {len(LOGS)}건 중 {len(logs)}건 성공")
rate = success / total
print(f"\n전체 파싱 성공률 {rate:.0%} ({success}/{total}) - 목표 100%: {'달성' if rate == 1 else '미달'}")

# 2) 정제 결과(표기 통일, 중복 기록 정리)
std = pd.DataFrame(results)
before = len(std)
std = std.drop_duplicates(["equip_id", "date", "cause"]).sort_values("equip_id")
print(f"\n[정제된 정비 대기 기록] {before}행 -> 중복 정리 후 {len(std)}행, 부대 {std['unit'].nunique()}곳")
print(std[["equip_id", "unit", "date", "wait_days", "cause", "source"]].to_string(index=False))

# 3) 요인 분류 정확도와 사람 확인 대상
chk = std[std["equip_id"].isin(TRUTH)]
acc = (chk["cause"] == chk["equip_id"].map(TRUTH)).mean()
print(f"\n[요인 분류] 정답이 있는 기록 {len(chk)}건 중 일치 {int((chk['cause'] == chk['equip_id'].map(TRUTH)).sum())}건 ({acc:.0%})")
print(std.groupby("cause").agg(건수=("equip_id", "size"), 평균대기일=("wait_days", "mean")).round(1).to_string())
print("사람 확인 대상('기타'로 분류):", ", ".join(f"{r.equip_id}('{r.reason[:14]}')" for r in std[std["cause"] == "기타"].itertuples()) or "없음")

# 4) 서식 변경 대응 시험 ①: 열 이름이 바뀐 새 양식 -> 별칭 사전 추가
new_form = "장비 코드,배속부대,지연 원인,지연 기간,접수 일자\nE-0031,가상B부대,부품 미입고,7일,2026.04.01\n"
ok, fail = load(new_form, "csv")
print(f"\n새 양식(열 이름 변경) 1차 시도: 성공 {len(ok)} / 실패 {len(fail)} -> {fail[0][1] if fail else '-'}")
for k, alias in {"equip": "장비 코드", "unit": "배속부대", "reason": "지연 원인", "wait": "지연 기간", "received": "접수 일자"}.items():
    COL_ALIAS[k].append(alias)
ok, fail = load(new_form, "csv")
print(f"별칭 사전 갱신 후: 성공 {len(ok)} / 실패 {len(fail)} (코드 수정 없이 설정만 변경)")

# 5) 서식 변경 대응 시험 ②: 새 표현의 사유 -> 분류 키워드 한 줄 추가
line = "가상C부대 E-0032 : 예비품 소진으로 8일간 정비 보류 (2026.04.02)"
r1 = parse_log(line)
CAUSE_KW[3][1].append("예비품")
r2 = parse_log(line)
print(f"새 사유 표현('예비품 소진') 1차 분류: {r1['cause']} -> 키워드 1개 추가 후: {r2['cause']} (코드 수정 없이 사전만 보완)")
print("※ 요인 분류는 키워드 기반 1차 분류이며, '기타'와 복합 사유는 정비 담당자가 확인해 확정한다.")


<details>
<summary><b>💡 [참고] 미션 02 예상 실행 결과 (Output)</b></summary>

```text
[CSV ] 6행 중 6행 성공
[TSV ] 5행 중 5행 성공
[JSON] 3행 중 3행 성공
[정비일지 문장] 5건 중 5건 성공

전체 파싱 성공률 100% (19/19) - 목표 100%: 달성

[정제된 정비 대기 기록] 19행 -> 중복 정리 후 18행, 부대 6곳
equip_id  unit       date  wait_days  cause source
  E-0001 가상A부대 2026-03-02        5.0  부품 대기    csv
  E-0002 가상B부대 2026-03-15       14.0 정비병 부족    csv
  E-0003 가상C부대 2026-01-10        2.0  외주 정비    csv
  E-0004 가상D부대 2026-02-27        3.0 검사·재작업    csv
  E-0005 가상E부대 2025-12-30       30.0  부품 대기    csv
  E-0006 가상F부대 2026-01-05        4.0 검사·재작업    csv
  E-0007 가상A부대 2026-02-11        6.0 정비병 부족    tsv
  E-0008 가상B부대 2026-01-20       10.0  부품 대기    tsv
  E-0009 가상C부대 2026-03-05        7.0  외주 정비    tsv
  E-0010 가상D부대 2025-11-03        3.0 검사·재작업    tsv
  E-0011 가상E부대 2026-02-01       14.0  부품 대기   json
  E-0012 가상F부대 2026-01-09        4.0 정비병 부족   json
  E-0013 가상A부대 2026-03-20        3.0 검사·재작업   json
  E-0021 가상A부대 2026-03-04        6.0  부품 대기    log
  E-0022 가상B부대 2026-03-05        3.0 정비병 부족    log
  E-0023 가상C부대 2026-03-06       14.0  외주 정비    log
  E-0024 가상D부대 2026-03-07        7.0 검사·재작업    log
  E-0025 가상E부대 2026-03-08        4.0     기타    log

[요인 분류] 정답이 있는 기록 18건 중 일치 18건 (100%)
        건수  평균대기일
cause            
검사·재작업   5    4.0
기타       1    4.0
부품 대기    5   13.0
외주 정비    3    7.7
정비병 부족   4    6.8
사람 확인 대상('기타'로 분류): E-0025('원인 불명 대기 4일')

새 양식(열 이름 변경) 1차 시도: 성공 0 / 실패 1 -> '필수 열 없음: reason'
별칭 사전 갱신 후: 성공 1 / 실패 0 (코드 수정 없이 설정만 변경)
... (이하 생략)
```
</details>
